# Student Performance Analysis and Prediction System

**Dataset note:** The bundled 600-row CSV is synthetic demonstration data, not real student records. Run all cells from the repository root.

## 1. Import libraries and load dataset

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score
ROOT=Path.cwd()
if not (ROOT/'data').exists(): ROOT=ROOT.parent
df=pd.read_csv(ROOT/'data/student_performance_dataset.csv')
print('Shape:',df.shape); display(df.head())

## 2. Inspect and clean data
The train/test split is made before fitting imputers or scalers, preventing preprocessing leakage.

In [ ]:
print(df.info()); display(df.describe().T)
print('Missing values:\n',df.isna().sum()); print('Duplicate IDs:',df.duplicated('student_id').sum())
df=df.drop_duplicates('student_id').copy()
features=['study_hours_per_day','attendance_rate','previous_exam_marks','assignment_score','class_participation','absences']
target='final_exam_marks'; X=df[features]; y=df[target]

## 3. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(8,5)); plt.hist(df[target],bins=20,edgecolor='black'); plt.title('Distribution of Final Marks'); plt.xlabel('Marks'); plt.ylabel('Students'); plt.show()
plt.figure(figsize=(8,5)); plt.scatter(df.study_hours_per_day,df[target],alpha=.6); plt.title('Study Hours vs Final Marks'); plt.xlabel('Study hours/day'); plt.ylabel('Final marks'); plt.show()
plt.figure(figsize=(8,5)); plt.scatter(df.attendance_rate,df[target],alpha=.6); plt.title('Attendance vs Final Marks'); plt.xlabel('Attendance (%)'); plt.ylabel('Final marks'); plt.show()
df['study_hour_group']=pd.cut(df.study_hours_per_day,bins=[0,2,4,6,9],labels=['0-2','2-4','4-6','6-9'])
df.boxplot(column=target,by='study_hour_group',figsize=(8,5)); plt.title('Final Marks by Study-Hour Group'); plt.suptitle(''); plt.show()
corr=df[features+[target]].corr(); plt.figure(figsize=(8,6)); plt.imshow(corr,vmin=-1,vmax=1); plt.xticks(range(len(corr)),corr.columns,rotation=45,ha='right'); plt.yticks(range(len(corr)),corr.columns); plt.colorbar(); plt.title('Correlation Heatmap'); plt.tight_layout(); plt.show()

## 4. Linear Regression and evaluation

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=42)
preprocess=ColumnTransformer([('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())]),features)])
reg=Pipeline([('preprocess',preprocess),('model',LinearRegression())]); reg.fit(X_train,y_train); pred=reg.predict(X_test)
mae=mean_absolute_error(y_test,pred); rmse=np.sqrt(mean_squared_error(y_test,pred)); r2=r2_score(y_test,pred); cv=cross_val_score(reg,X_train,y_train,cv=5,scoring='r2')
print(f'MAE={mae:.3f}; RMSE={rmse:.3f}; R2={r2:.3f}; CV R2={cv.mean():.3f} +/- {cv.std():.3f}')
plt.figure(figsize=(7,6)); plt.scatter(y_test,pred,alpha=.7); plt.plot([0,100],[0,100],'--'); plt.xlabel('Actual'); plt.ylabel('Predicted'); plt.title('Actual vs Predicted'); plt.show()
display(pd.DataFrame({'Actual':y_test.values,'Predicted':pred.round(1)}).head(15))

## 5. Optional pass/fail classification
The 40/100 threshold is an illustrative project assumption, not a universal grading policy.

In [ ]:
y_pass=(y>=40).astype(int)
Xtr,Xte,ytr,yte=train_test_split(X,y_pass,test_size=.2,random_state=42,stratify=y_pass)
clf=Pipeline([('preprocess',preprocess),('model',LogisticRegression(max_iter=1000,class_weight='balanced'))]); clf.fit(Xtr,ytr); cp=clf.predict(Xte)
print('Accuracy',accuracy_score(yte,cp),'Precision',precision_score(yte,cp,zero_division=0),'Recall',recall_score(yte,cp,zero_division=0),'F1',f1_score(yte,cp,zero_division=0))

## 6. Findings, limitations and future improvements
Interpret charts as associations, not causation. MAE is average absolute error; RMSE penalizes larger errors; R² compares the model with a mean baseline. Because the bundled dataset is synthetic, results are illustrative only. Future work should use verified authorized records, compare multiple models, tune with validation data, and assess fairness and privacy.